In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("catalogo", "bank_dev")
dbutils.widgets.text("esquema_bronze", "bronze")
dbutils.widgets.text("esquema_silver", "silver")
dbutils.widgets.text("storageName", "saprojectbankdeveastus")
dbutils.widgets.text("containerSilver", "silver")

In [0]:
catalogo         = dbutils.widgets.get("catalogo")
esquema_bronze   = dbutils.widgets.get("esquema_bronze")
esquema_silver   = dbutils.widgets.get("esquema_silver")
storageName      = dbutils.widgets.get("storageName")
containerSilver  = dbutils.widgets.get("containerSilver")

In [0]:
checkpoint_location_s = f"abfss://{containerSilver}@{storageName}.dfs.core.windows.net/_checkpoints/users" 


tabla_bronze_users = f"{catalogo}.{esquema_bronze}.users"
tabla_silver_users = f"{catalogo}.{esquema_silver}.dim_users"

In [0]:
dbutils.fs.rm(f"abfss://{containerSilver}@{storageName}.dfs.core.windows.net/_checkpoints/users", True)

In [0]:
_df = spark.readStream.table(tabla_bronze_users)

In [0]:
# ===================== TRANSFORMACIÓN =====================
df_silver_users = (
    _df \
    .withColumn("birth_date", F.to_date(F.concat(
        F.col("birth_year").cast("string"), 
        F.lit("-"), 
        F.format_string("%02d", F.col("birth_month").cast("int")), 
        F.lit("-01")
    ))) \
    .withColumn("per_capita_income", F.col("per_capita_income").cast("decimal(15,2)")) \
    .withColumn("yearly_income", F.col("yearly_income").cast("decimal(15,2)")) \
    .withColumn("total_debt", F.col("total_debt").cast("decimal(15,2)")) \
    # .withColumn("valid_from", F.current_timestamp()) \
    .withColumn("valid_from", F.to_timestamp(F.lit("1900-01-01 00:00:00")))
    .withColumn("valid_to", F.to_timestamp(F.lit("9999-12-31 23:59:59"))) \
    .withColumn("is_current", F.lit(True)) \
    .withColumn("silver_load_date", F.current_timestamp()) \
    .drop("birth_year", "birth_month", "current_age", "retirement_age")

)

In [0]:
# ===================== CARGA A SILVER (STREAMING + MERGE SCD2) =====================

def upsert_users(df_batch, batch_id):
    if df_batch.isEmpty():
        return
        
    print(f"Procesando lote {batch_id}...")
    df_batch.createOrReplaceTempView("stg_users")

    # 🛠️ VERIFICACIÓN
    table_exists = spark.catalog.tableExists(tabla_silver_users)

    if not table_exists:
        print(f"La tabla {tabla_silver_users} no existe. Creando carga inicial...")
        (df_batch.write
         .format("delta")
         .mode("overwrite")
         .saveAsTable(tabla_silver_users))
    else:
        # Usamos df_batch.sparkSession para garantizar contexto en el Stream
        df_batch.sparkSession.sql(f"""
        MERGE INTO {tabla_silver_users} AS target
        USING (
            SELECT id AS merge_key, stg.* FROM stg_users stg
            UNION ALL
            SELECT NULL AS merge_key, stg.*
            FROM stg_users stg
            JOIN {tabla_silver_users} tgt ON stg.id = tgt.id AND tgt.is_current = true
            WHERE NOT (stg.address <=> tgt.address)
               OR NOT (stg.yearly_income <=> tgt.yearly_income)
               OR NOT (stg.total_debt <=> tgt.total_debt)
        ) AS staged_updates
        ON target.id = staged_updates.merge_key

        WHEN MATCHED AND target.is_current = true 
          AND (NOT (target.address <=> staged_updates.address)
               OR NOT (target.yearly_income <=> staged_updates.yearly_income)
               OR NOT (target.total_debt <=> staged_updates.total_debt)) THEN 
          UPDATE SET target.is_current = false, target.valid_to = current_timestamp()

        WHEN NOT MATCHED THEN 
          INSERT *
        """)

# Lanzamos el proceso de Streaming
query = (df_silver_users.writeStream
         .foreachBatch(upsert_users)
         .option("checkpointLocation", checkpoint_location_s) 
         .trigger(availableNow=True) 
         .start())